In [5]:
##### Global scale: capital, labor, production in conservation priority areas
# calculates absolute and % of global total inside priority areas (additional vs already protected)

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import rasterio
import xarray as xr

In [6]:
##### SET-UP

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from helpers.reproject_raster import reproject_raster, array_to_raster

OUT_DIR = ROOT / "Results" / "30x30"
OUT_DIR.mkdir(parents=True, exist_ok=True)

##### LOAD DATA

# capital rasters
capital_path = ROOT / "Results/Raster_models/rescaled_capital_USD.tif"
capital_path_p10 = ROOT / "Results/Raster_models/rescaled_capital_USD_p10.tif"
capital_path_p90 = ROOT / "Results/Raster_models/rescaled_capital_USD_p90.tif"

# labor rasters
labor_path = ROOT / "Results/Raster_models/rescaled_jobs.tif"
labor_path_p10 = ROOT / "Results/Raster_models/rescaled_jobs_p10.tif"
labor_path_p90 = ROOT / "Results/Raster_models/rescaled_jobs_p90.tif"

# production raster
production_path = ROOT / "Data/total_production_tonnes_2020.tif"

# conservation scenario raster
# 0 = not in top 30%, 1 = in top 30%, not PA, 2 = in top 30%, already PA
biodiversity_with_PA = ROOT / "Data/conservation_scenario.tif"

In [7]:
##### DATA PREP

# resample to match conservation scenario (equal area projection)
def load_summed(path):
    return array_to_raster(reproject_raster(str(path), resampling="sum"))

production = load_summed(production_path)
capital = load_summed(capital_path)
capital_p10 = load_summed(capital_path_p10)
capital_p90 = load_summed(capital_path_p90)
labor = load_summed(labor_path)
labor_p10 = load_summed(labor_path_p10)
labor_p90 = load_summed(labor_path_p90)

# rasters with uncertainty estimates
ci_rasters = {
    "Capital": {"p10": capital_p10, "p90": capital_p90},
    "Labor":   {"p10": labor_p10,   "p90": labor_p90},
}

# observed/model values
value_rasters = {
    "Capital": capital,
    "Labor": labor,
    "Production": production,
}

def read_array(src, band=1):
    if isinstance(src, xr.DataArray):
        if "band" in src.dims:
            src = src.isel(band=band - 1)
        return src.squeeze().values.astype(float)

    with rasterio.open(src) as r:
        arr = r.read(band).astype(float)
        nodata = r.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
    return arr

priority_arr = read_array(biodiversity_with_PA)
in_priority = (priority_arr == 1) | (priority_arr == 2)

In [9]:
##### GLOBAL SUMMARY TABLE

summary_rows = []

for variable, raster in value_rasters.items():
    value_arr = read_array(raster)
    global_total = np.nansum(value_arr)

    additional = np.nansum(value_arr[priority_arr == 1])
    already = np.nansum(value_arr[priority_arr == 2])
    total = additional + already

    if variable in ci_rasters:
        lo_arr = read_array(ci_rasters[variable]["p10"])
        hi_arr = read_array(ci_rasters[variable]["p90"])
        total_p10 = np.nansum(lo_arr[in_priority])
        total_p90 = np.nansum(hi_arr[in_priority])
    else:
        total_p10 = total_p90 = total

    summary_rows.append({
        "Variable": variable,
        "Additional_priority": additional,
        "Already_in_PA": already,
        "Total_priority": total,
        "Pct_additional": 100 * additional / global_total,
        "Pct_already": 100 * already / global_total,
        "Pct_total": 100 * total / global_total,
        "Pct_total_p10": 100 * total_p10 / global_total,
        "Pct_total_p90": 100 * total_p90 / global_total,
        "Global_total": global_total,
    })

summary_df = pd.DataFrame(summary_rows)

##### SAVE
summary_df.to_csv(OUT_DIR / "global_exposure_summary.csv", index=False)